In [1]:
import os, sys, numpy as np

print("Python:", sys.version.split()[0])
print("CWD:", os.getcwd())

Python: 3.10.12
CWD: /local0/home/mfilo/git/CRN-GenerativeAI/apps


In [2]:
from RL4CRN.utils.input_interface import (
    Configurator, SolverCfg,
    make_task,
    make_session_and_trainer,
    print_task_summary,
    run_smoke_reward,
)

In [3]:
from RL4CRN.utils.crn_builders import build_simple_IOCRN

# choose preset
cfg = Configurator.preset("paper")

# select simulator and set tolerances
cfg.solver.algorithm = "CVODE"
cfg.solver.rtol = 1e-10
cfg.solver.atol = 1e-10

# build template IO/CRN
species_labels = ['X_1', 'X_2', 'X_3']
crn, species_labels = build_simple_IOCRN(
    species=species_labels,
    input_map={"X_1": "u_1"},
    dilution_map={},
    output_species="X_3",
    solver=cfg.solver,
)

from RL4CRN.utils.library_builders import build_MAK_library

# library components
library_components = build_MAK_library(crn, species_labels, order=cfg.library.order)

In [4]:
periods = [10, 15, 20]
task = make_task(
    template_crn=crn,
    library_components=library_components,
    kind="oscillator_freq",
    species_labels=species_labels,
    n_inputs=1,
    u_values=[np.array([1/p]) for p in periods],
    ic=("constant", 0.01),
    osc_w=[0/10, 6/10, 1/10, 3/10],                              # [mean, frequency, damping, periodicity index]
    t_f=100, n_t=1000
)
print_task_summary(task)

Task: oscillator_freq
time_horizon: (1000,) [0..100.0]
num scenarios: 3
first 3 u: [array([[0.1]], dtype=float32), array([[0.06666667]], dtype=float32), array([[0.05]], dtype=float32)]



In [7]:
# ---- Task config (logic) ----
cfg.task = task

# ---- Train config ----
cfg.train.max_added_reactions = 5
cfg.train.epochs = 30
cfg.train.render_every = 5
cfg.train.seed = 0

# ---- Library / solver (advanced knobs) ----
cfg.library.order = 2
cfg.library.include_dilution = False

In [8]:
session, trainer = make_session_and_trainer(cfg)

AttributeError: 'TaskSpec' object has no attribute 'N_t'

In [ ]:
cfg.describe()